<div style="padding:22px 24px;border:1px solid #d8e0e8;border-radius:18px;background:#fbfcfe;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<div style="display:inline-block;padding:6px 10px;border-radius:999px;background:#e7eef7;color:#35506b;font-size:12px;font-weight:600;letter-spacing:0.04em;text-transform:uppercase;">Chapter 3 notebook</div>
<h1 style="margin:12px 0 8px 0;color:#182b3c;font-size:32px;line-height:1.15;">Giving Agents Hands: tools, function calling, and MCP</h1>
<p style="margin:0;color:#4d6277;font-size:15px;max-width:840px;line-height:1.6;">This notebook follows Chapter 3 in reading order. Each section tells you which part of the chapter you are looking at, what you are creating, and which code matters.</p>
</div>


## Start Here: how to use this notebook

This is the reader-facing path for Chapter 3. It is not a replacement for the prose; it is the practical map that keeps the prose, diagrams, and code in one place.

### What you will build

1. Tool interfaces that the model can select reliably.
2. Error responses that help an agent recover instead of retrying blindly.
3. Idempotent write tools that are safe under retries.
4. Approval gates for serious side effects.
5. MCP server and client shapes.
6. A financial services audit-prep agent shell with auditable tool calls.

### One-time setup

```bash
cd "chapter 3"
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
jupyter notebook
```

Most cells run offline. The notebook uses local teaching doubles for external systems such as ERP, payments, approvals, ledger data, the Agents SDK, and MCP. That keeps the examples runnable while preserving the shape of the production code.


In [ ]:
from __future__ import annotations

# This first cell is setup. You do not need to understand every line before
# reading the chapter examples. Its job is to create safe local stand-ins for
# external systems that would normally live outside the notebook.

import asyncio
import hashlib
import inspect
import sys
import types
from dataclasses import dataclass, field as dataclass_field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Literal, Optional

from pydantic import BaseModel, Field

chapter_root = Path.cwd()
print(f"Notebook root: {chapter_root}")

# The real Agents SDK uses @function_tool to mark a Python function as a tool.
# This teaching version only stores the information we need in the notebook:
# the tool name and the function docstring.
def function_tool(fn: Callable | None = None, **tool_kwargs: Any):
    def decorate(func: Callable):
        func.is_function_tool = True
        func.tool_name = tool_kwargs.get("name", func.__name__)
        func.tool_description = inspect.getdoc(func) or ""
        return func
    return decorate(fn) if fn is not None else decorate

# These small dataclasses stand in for SDK objects. They keep the examples
# readable because readers can inspect their fields directly in the notebook.
@dataclass
class MCPServerStreamableHttp:
    name: str
    params: dict[str, Any]

@dataclass
class Agent:
    name: str
    instructions: str
    mcp_servers: list[Any] = dataclass_field(default_factory=list)
    output_type: Any | None = None
    output_guardrails: list[Callable] = dataclass_field(default_factory=list)

@dataclass
class RunResult:
    final_output: Any
    steps: list[str]

class Runner:
    @staticmethod
    async def run(agent: Agent, input: str) -> RunResult:
        # A real runner would call the model and tools. This mock returns the
        # shape of a successful run so the rest of the notebook can stay local.
        server_names = [server.name for server in agent.mcp_servers]
        return RunResult(
            final_output={
                "agent": agent.name,
                "servers": server_names,
                "summary": "Mock run complete. Replace the teaching doubles with the real SDK for production.",
                "input": input,
            },
            steps=["discover_tools", "plan", "call_tool", "summarize"],
        )

class FastMCP:
    def __init__(self, name: str):
        self.name = name
        self.tools: dict[str, Callable] = {}
        self.resources: dict[str, Callable] = {}

    def tool(self):
        # @app.tool() means this function becomes callable through MCP.
        def decorate(func: Callable):
            self.tools[func.__name__] = func
            return func
        return decorate

    def resource(self, uri: str):
        # @app.resource(...) means this is read-only context, not an action.
        def decorate(func: Callable):
            self.resources[uri] = func
            return func
        return decorate

    def run(self, **kwargs: Any):
        # In this notebook, starting the server only prints what would happen.
        # That prevents an accidental long-running process inside Jupyter.
        print(f"FastMCP teaching server '{self.name}' would run with: {kwargs}")

# The next block makes imports such as `from agents import Agent` work even on
# a machine that has not installed the real SDK yet. In production code, delete
# this whole sys.modules section and install/import the real packages instead.
agents_mod = types.ModuleType("agents")
agents_mod.function_tool = function_tool
agents_mod.Agent = Agent
agents_mod.Runner = Runner
sys.modules["agents"] = agents_mod

agents_mcp_mod = types.ModuleType("agents.mcp")
agents_mcp_mod.MCPServerStreamableHttp = MCPServerStreamableHttp
sys.modules["agents.mcp"] = agents_mcp_mod

mcp_mod = types.ModuleType("mcp")
mcp_server_mod = types.ModuleType("mcp.server")
mcp_fastmcp_mod = types.ModuleType("mcp.server.fastmcp")
mcp_fastmcp_mod.FastMCP = FastMCP
sys.modules["mcp"] = mcp_mod
sys.modules["mcp.server"] = mcp_server_mod
sys.modules["mcp.server.fastmcp"] = mcp_fastmcp_mod

# SQLModel is only used to show the journal shape. This tiny stand-in lets the
# class definition run without requiring a database or SQLAlchemy setup.
class SQLModel:
    def __init_subclass__(cls, **kwargs: Any):
        return None

    def __init__(self, **data: Any):
        for key, value in data.items():
            setattr(self, key, value)

def SQLField(default: Any = None, **kwargs: Any):
    return default

sqlmodel_mod = types.ModuleType("sqlmodel")
sqlmodel_mod.SQLModel = SQLModel
sqlmodel_mod.Field = SQLField
sys.modules["sqlmodel"] = sqlmodel_mod

# Everything below is a fake enterprise system. The tools call these objects so
# readers can see realistic inputs and outputs without ERP, payments, approval,
# or ledger infrastructure.
class MockERPClient:
    def __init__(self):
        self.submissions: dict[str, dict[str, Any]] = {}

    def preview(self, proposal: dict[str, Any]) -> dict[str, Any]:
        return {
            "ok": True,
            "side_effect": False,
            "po_number": proposal["po_number"],
            "line_item": proposal["line_item"],
            "new_quantity": proposal["new_quantity"],
            "estimated_total_delta": proposal["new_quantity"] * 125,
            "warnings": [],
        }

    def submit(self, proposal: dict[str, Any], idempotency_key: str) -> dict[str, Any]:
        replayed = idempotency_key in self.submissions
        if not replayed:
            self.submissions[idempotency_key] = {
                "approval_request_id": f"APR-{len(self.submissions) + 1:04d}",
                "status": "pending",
                "proposal": proposal,
            }
        return {"ok": True, "replayed": replayed, **self.submissions[idempotency_key]}

    def status(self, approval_request_id: str) -> dict[str, Any]:
        for submission in self.submissions.values():
            if submission["approval_request_id"] == approval_request_id:
                return {"ok": True, "approval_request_id": approval_request_id, "status": submission["status"]}
        return {"ok": False, "error_class": "not_found", "retryable": False}

class MockPayments:
    def __init__(self):
        self.refunds_by_key: dict[str, dict[str, Any]] = {}

    def find_refund_by_key(self, idempotency_key: str) -> dict[str, Any] | None:
        return self.refunds_by_key.get(idempotency_key)

    def create_refund(self, *, charge_id: str, amount_cents: int, idempotency_key: str, metadata: dict[str, Any]) -> dict[str, Any]:
        refund = {
            "refund_id": f"RF-{len(self.refunds_by_key) + 1:05d}",
            "charge_id": charge_id,
            "amount_cents": amount_cents,
            "idempotency_key": idempotency_key,
            "metadata": metadata,
        }
        self.refunds_by_key[idempotency_key] = refund
        return refund

class MockApprovalService:
    def __init__(self):
        self.requests: dict[str, dict[str, Any]] = {}

    async def create(self, payload: dict[str, Any]) -> str:
        request_id = f"HITL-{len(self.requests) + 1:04d}"
        self.requests[request_id] = payload
        return request_id

    async def wait(self, request_id: str, timeout_hours: int) -> dict[str, Any]:
        await asyncio.sleep(0)
        return {
            "ok": True,
            "request_id": request_id,
            "decision": "approved",
            "approval_token": f"token-for-{request_id}",
            "timeout_hours": timeout_hours,
        }

class MockLedger:
    def snapshot(self, **query: Any) -> dict[str, Any]:
        return {
            "period": query["period"],
            "entity": query["entity"],
            "materiality_threshold": query["materiality_threshold"],
            "accounts": [
                {"account": "1000", "name": "Cash", "balance": 2_450_000},
                {"account": "2100", "name": "Short-term liabilities", "balance": -910_000},
            ],
        }

    def account_dictionary_yaml(self) -> str:
        return "1000: Cash\n2100: Short-term liabilities\n"

_erp_client = MockERPClient()
_payments = MockPayments()
_approval_service = MockApprovalService()
_ledger = MockLedger()

# Shared values used later in the notebook.
AUDIT_PREP_INSTRUCTIONS = "Assemble audit evidence from approved finance systems."
EVIDENCE_AGENT_INSTRUCTIONS = "Prepare evidence bundles, keep draft work reversible, and publish only after approval."

data_server = MCPServerStreamableHttp(name="finance-data", params={"url": "https://example.invalid/data"})
policy_server = MCPServerStreamableHttp(name="policy", params={"url": "https://example.invalid/policy"})
evidence_writer = MCPServerStreamableHttp(name="evidence-store", params={"url": "https://example.invalid/evidence"})

def bundle_completeness_check(output: Any) -> bool:
    return True

print("Teaching doubles loaded.")


## 1. Understanding what a tool actually is

Chapter section: **Understanding what a tool actually is**.

What you are creating here is the smallest useful mental model: a tool is a Python function plus a typed interface and a description. The model does not execute the function. The runtime executes the function after the model emits a structured request.


In [ ]:
# A normal Python function can become a tool once the agent runtime knows about it.
def get_current_weather(city: str) -> dict:
    """Return the current weather for a city."""
    return {"city": city, "temperature_c": 18, "condition": "clear"}

# The model does not run Python. It emits a structured request like this:
#   1. which tool it wants
#   2. which arguments it wants to pass
model_emitted_tool_call = {
    "tool_name": "get_current_weather",
    "arguments": {"city": "Lisbon"},
}

# The runtime owns the tool registry. This is the lookup table from a tool name
# to the real Python function that should run.
tool_registry = {"get_current_weather": get_current_weather}

# The runtime selects the function and passes the model-provided arguments.
selected_tool = tool_registry[model_emitted_tool_call["tool_name"]]
selected_tool(**model_emitted_tool_call["arguments"])


## 2. How the agent picks the right tool

Chapter section: **How the agent picks the right tool**.

What you are creating here is a tool surface the model can navigate. The chapter's key point is decision reliability: the model must choose the right tool, not merely produce valid JSON for the wrong one.


In [ ]:
# Tool descriptions are part of the interface. A vague description makes the
# model guess. A precise description tells the model when the tool is relevant.
tool_descriptions = {
    "get_user_profile": "Use when the user asks for account identity, contact, KYC, or profile details.",
    "search_orders": "Use when the user asks about purchases, order history, shipment status, or invoices.",
    "send_email": "Use only after the user asks to send a specific email message to a specific recipient.",
}

user_message = "Tell me about my account and whether my mailing address is current."

# This scoring function is deliberately simple. Real models do not count words
# this way, but the learning point is useful: the user's intent has to match the
# words and boundaries you give each tool.
def rough_match_score(message: str, description: str) -> int:
    message_words = set(message.lower().replace(".", "").split())
    description_words = set(description.lower().replace(",", "").replace(".", "").split())
    return len(message_words & description_words)

# The best tool should rise to the top. If the wrong tool wins in your real app,
# first improve tool names, descriptions, and boundaries before adding prompts.
ranked_tools = sorted(
    tool_descriptions.items(),
    key=lambda item: rough_match_score(user_message, item[1]),
    reverse=True,
)

ranked_tools


## 3. Designing tools the model can actually use

Chapter section: **Designing tools the model can actually use**.

What you are creating here is a set of narrow, intent-shaped purchase-order tools. This is the chapter's concrete contrast with broad endpoint-shaped tools such as `erp_action(action, entity, params)`.


In [ ]:
from agents import function_tool
from pydantic import BaseModel, Field
from typing import Literal

# Pydantic models are useful because they make the tool contract explicit.
# The model must provide a PO number, a line item, a quantity, and one valid
# reason code. This reduces ambiguity before the tool touches a backend system.
class POChangeProposal(BaseModel):
    po_number: str = Field(description="Purchase order identifier")
    line_item: int = Field(ge=1, description="Line item position")
    new_quantity: int = Field(ge=0)
    reason_code: Literal["price_correction", "vendor_change", "cancel_line"]

@function_tool
def preview_po_change(proposal: POChangeProposal) -> dict:
    """Show what a purchase-order change would do without making it.
    Returns updated totals, tax impact, and any validation warnings.
    Safe to call as many times as you like. No side effects."""
    # Preview tools should not write. They are safe for demos, retries, and review.
    return _erp_client.preview(proposal.model_dump())

@function_tool
def submit_po_change_for_approval(proposal: POChangeProposal,
                                  idempotency_key: str) -> dict:
    """Submit a previewed change for approval. The change does not
    take effect until an approver acts. Returns an approval_request_id
    that you can poll for status."""
    # The idempotency key tells the backend this is the same business intent if
    # the agent retries. Same intent, same key. New intent, new key.
    return _erp_client.submit(proposal.model_dump(), idempotency_key)

@function_tool
def get_po_change_status(approval_request_id: str) -> dict:
    """Check the status of a previously submitted approval request.
    Status is one of: pending, approved, rejected, expired."""
    # Status checks are read-only, so they are safe to repeat.
    return _erp_client.status(approval_request_id)


In [ ]:
# First create a valid proposal. If a field is missing or invalid, Pydantic will
# fail before the tool runs. That is exactly what we want for tool inputs.
proposal = POChangeProposal(
    po_number="PO-1883",
    line_item=2,
    new_quantity=4,
    reason_code="price_correction",
)

# The safe order is: preview, submit for approval, then check status.
preview = preview_po_change(proposal)
submitted = submit_po_change_for_approval(proposal, idempotency_key="RUN-42:po-change:PO-1883:2")
status = get_po_change_status(submitted["approval_request_id"])

preview, submitted, status


## 4. Designing errors for the agent

Chapter section: **Design errors for the agent, not for a stack trace**.

What you are creating here is an error object with enough information for the agent loop, the developer, and the end user. This is how you break retry loops caused by vague failures.


In [ ]:
# A good tool error is written for three readers:
#   - the agent loop, which needs retryable and machine_action_hint
#   - the end user, who needs user_safe_message
#   - the developer, who needs developer_detail
agent_readable_error = {
    "ok": False,
    "error_class": "rate_limited",
    "retryable": True,
    "retry_after_seconds": 30,
    "user_safe_message": "The vendor system is busy. I can try again "
                         "in about thirty seconds.",
    "developer_detail": "HTTP 429 from upstream; rate-limit-remaining=0",
    "machine_action_hint": "backoff_then_retry"
}

agent_readable_error


## 5. Engineering the side-effect boundary

Chapter section: **Engineering the side-effect boundary**.

What you are creating here is the chapter's safety boundary for actions that change the world. The core production idea is idempotency: a retried write should be safe because the business intent has a stable key.

![Figure 3.1 - The stochastic loop of death and two ways to break it](assets/figure_3_1_stochastic_loop.png)


In [ ]:
from agents import function_tool
from pydantic import BaseModel, Field

# Refunds are write actions. That makes the input contract more important than
# it would be for a simple read. The amount must be positive, and the reason is
# capped so a model cannot send an unbounded blob of text.
class RefundRequest(BaseModel):
    customer_id: str
    original_charge_id: str
    amount_cents: int = Field(gt=0)
    reason: str = Field(max_length=200)

@function_tool
def issue_refund(req: RefundRequest, idempotency_key: str) -> dict:
    """Issue a refund against a previously authorized charge.
    The idempotency_key must be stable across retries for the same
    business intent. A duplicate key returns the original result
    without issuing a second refund."""
    # First look for an existing result under the same key. This is the critical
    # safe-replay check. It prevents duplicate refunds after timeouts or retries.
    existing = _payments.find_refund_by_key(idempotency_key)
    if existing is not None:
        return {"ok": True, "replayed": True, **existing}

    # Only create a refund when this business intent has not already succeeded.
    result = _payments.create_refund(
        charge_id=req.original_charge_id,
        amount_cents=req.amount_cents,
        idempotency_key=idempotency_key,
        metadata={"reason": req.reason, "customer_id": req.customer_id},
    )
    return {"ok": True, "replayed": False, **result}


In [ ]:
refund_request = RefundRequest(
    customer_id="CUST-1007",
    original_charge_id="CH-5551",
    amount_cents=12_500,
    reason="duplicate billing event",
)

# This key represents the user's business intent. It must stay the same when the
# agent retries the same refund. Do not generate a fresh random key on retry.
stable_key = "RUN-9001:refund:CUST-1007:CH-5551:12500"

first_attempt = issue_refund(refund_request, stable_key)
retry_attempt = issue_refund(refund_request, stable_key)

# The second result should replay the first refund, not create a second refund.
first_attempt, retry_attempt


## 6. Approval gates

Chapter section: **Approval gates, where the human says yes**.

What you are creating here is a discrete checkpoint. The proposed action is persisted, a reviewer approves or rejects it, and the agent resumes with an approval token instead of an implied permission.


In [ ]:
from agents import function_tool
from pydantic import BaseModel
from typing import Literal

# Approval requests should include enough information for a reviewer to know
# exactly what they are approving. The proposal hash binds approval to a specific
# proposed effect, not to a vague instruction like "publish the bundle".
class ApprovalRequest(BaseModel):
    action_kind: Literal["refund", "po_change", "evidence_publish"]
    proposal_hash: str
    summary: str
    requested_by_run_id: str

@function_tool
async def request_human_approval(req: ApprovalRequest) -> dict:
    """Create an approval request and suspend the run until a
    reviewer approves or rejects. Returns an approval_token on
    approval, or a rejection reason."""
    # In production, most systems do not wait inside the tool for 72 hours.
    # They park the run, store request_id, and resume when the reviewer acts.
    request_id = await _approval_service.create(req.model_dump())
    decision = await _approval_service.wait(request_id, timeout_hours=72)
    return decision


In [ ]:
# This example asks for approval to publish one specific evidence bundle.
# The token returned by the approval service is what later permits the commit.
approval = await request_human_approval(
    ApprovalRequest(
        action_kind="evidence_publish",
        proposal_hash="sha256:8c0ffee",
        summary="Publish Q2 liquidity-reporting evidence bundle for CTRL-77.",
        requested_by_run_id="RUN-9001",
    )
)

approval


## 7. Plugging in with the Model Context Protocol

Chapter section: **Plugging in with the Model Context Protocol**.

What you are creating here is the MCP shape: a host with clients connected to servers that expose tools, resources, and prompts. In the notebook, the server is a local teaching double. In production, this is where the real MCP server process or remote endpoint sits.

![Figure 3.2 - One host, three clients, three servers](assets/figure_3_2_mcp_topology.png)


In [ ]:
from mcp.server.fastmcp import FastMCP
from pydantic import BaseModel, Field

# An MCP server exposes capabilities to a host application. In this example,
# finance-evidence is the server that owns finance evidence data.
app = FastMCP("finance-evidence")

# This is the typed input for one MCP tool. The pattern rejects invalid periods,
# the entity must not be empty, and materiality cannot be negative.
class TrialBalanceQuery(BaseModel):
    period: str = Field(pattern=r"^\d{4}-Q[1-4]$")
    entity: str = Field(min_length=2)
    materiality_threshold: int = Field(ge=0)

@app.tool()
def get_trial_balance_snapshot(query: TrialBalanceQuery) -> dict:
    """Return a trial balance snapshot for the named period and
    entity, filtered to accounts above the materiality threshold.
    Read-only. Safe to call repeatedly."""
    return _ledger.snapshot(**query.model_dump())

@app.resource("schema://accounts")
def accounts_schema() -> str:
    # Resources are read-only context. They are good for schemas, policies, and
    # reference data that the host can read without asking the model to call a tool.
    return _ledger.account_dictionary_yaml()

if __name__ == "__main__":
    # The notebook mock only prints the server settings. A real MCP server would
    # keep running here and listen for client connections.
    app.run(transport="streamable-http", host="0.0.0.0", port=8765)


In [ ]:
# Create a valid query, then inspect what the MCP server has registered.
query = TrialBalanceQuery(period="2026-Q2", entity="Entity 14", materiality_threshold=100_000)

registered_tools = list(app.tools)
registered_resources = list(app.resources)
snapshot = get_trial_balance_snapshot(query)
schema_text = accounts_schema()

registered_tools, registered_resources, snapshot, schema_text


In [ ]:
from agents import Agent, Runner
from agents.mcp import MCPServerStreamableHttp

# This object points the agent runtime at a remote MCP server. The URL is a
# placeholder here. In a real environment, it would be your internal endpoint.
evidence_server = MCPServerStreamableHttp(
    name="finance-evidence",
    params={"url": "https://internal-mcp.bank.local/finance-evidence"},
)

# The agent receives MCP servers as part of its tool surface.
agent = Agent(
    name="audit-prep-assistant",
    instructions=AUDIT_PREP_INSTRUCTIONS,
    mcp_servers=[evidence_server],
)

# The mock runner returns the shape of a run. With the real SDK, this is where
# the model would plan, call MCP tools, and produce a final response.
result = await Runner.run(
    agent,
    input="Pull the Q2 trial balance for Entity 14 above materiality "
          "and summarize movements over five percent.",
)

result.final_output


## 8. Building a financial services audit-prep agent

Chapter section: **Building a financial services audit-prep agent**.

What you are creating here is the realistic enterprise workflow: read from finance systems, build a draft evidence bundle, journal every tool call, request human approval, and publish only after the approval token is bound to the exact proposed bundle.

![Figure 3.3 - High-level component diagram for the audit-prep agent](assets/figure_3_3_audit_prep_architecture.png)


In [ ]:
from sqlmodel import SQLModel, Field
from typing import Optional
from datetime import datetime

# The journal is the audit trail for tool calls. It lets a resumed run know what
# already happened, and it gives reviewers a clear record of each side effect.
class ToolCallJournal(SQLModel, table=True):
    id: int = Field(primary_key=True)
    run_id: str = Field(index=True)             # Which agent run made the call.
    tool_call_id: str = Field(unique=True)      # Unique id for this one tool call.
    tool_name: str                              # Which tool was called.
    idempotency_key: str = Field(index=True)    # Stable key for safe retries.
    status: str                                 # pending, succeeded, failed, approved, etc.
    result_hash: Optional[str] = None           # Fingerprint of the tool result.
    materialized_object_id: Optional[str] = None # External object created by the tool.
    approval_reference: Optional[str] = None    # Human approval request id, if needed.
    requested_at: datetime
    completed_at: Optional[datetime] = None

ToolCallJournal.__annotations__


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# This is the user's request in structured form. The model does not get to invent
# arbitrary periods or jurisdictions; the schema defines what is valid.
class EvidenceRequest(BaseModel):
    control_id: str
    period: str = Field(pattern=r"^\d{4}-Q[1-4]$")
    jurisdiction: Literal["uk", "us", "eu"]
    requested_by_user_id: str

# This is the final status shape. It keeps downstream code from parsing prose.
class EvidenceBundleStatus(BaseModel):
    bundle_id: str
    status: Literal["draft", "awaiting_approval",
                    "approved", "published", "failed"]
    artifact_count: int
    awaiting_reviewer_id: Optional[str] = None


In [ ]:
# This is the chapter's agent shell. The important part is the boundary:
# read-oriented MCP servers are separated from the evidence writer, and the
# output has a typed shape that guardrails can check.
agent = Agent(
    name="audit-evidence-assembler",
    instructions=EVIDENCE_AGENT_INSTRUCTIONS,
    mcp_servers=[data_server, policy_server, evidence_writer],
    output_type=EvidenceBundleStatus,
    output_guardrails=[bundle_completeness_check],
)

agent


In [ ]:
def deterministic_idempotency_key(run_id: str, business_intent: str) -> str:
    # A deterministic key is repeatable. The same run and same business intent
    # produce the same key, which is what makes retries safe.
    digest = hashlib.sha256(f"{run_id}:{business_intent}".encode("utf-8")).hexdigest()[:16]
    return f"{run_id}:{digest}"

def result_hash(payload: dict[str, Any]) -> str:
    # A hash gives the approval gate a compact fingerprint of the proposed output.
    # In production, use canonical JSON rather than repr for cross-language safety.
    return hashlib.sha256(repr(sorted(payload.items())).encode("utf-8")).hexdigest()

async def assemble_evidence_bundle(req: EvidenceRequest) -> dict[str, Any]:
    # The run id groups every tool call and journal entry for this workflow.
    run_id = f"RUN-{req.control_id}-{req.period}"
    bundle_id = f"BND-{req.control_id}-{req.period}"

    # The business intent is the thing we want to protect from duplicate writes.
    publish_intent = f"publish:{bundle_id}:{req.requested_by_user_id}"
    publish_key = deterministic_idempotency_key(run_id, publish_intent)

    # The bundle starts as a draft. Draft work is easier to rerun and inspect.
    draft_status = EvidenceBundleStatus(
        bundle_id=bundle_id,
        status="awaiting_approval",
        artifact_count=3,
        awaiting_reviewer_id="reviewer-01",
    )

    # Publishing crosses the side-effect boundary, so it needs explicit approval.
    approval = await request_human_approval(
        ApprovalRequest(
            action_kind="evidence_publish",
            proposal_hash=result_hash(draft_status.model_dump()),
            summary=f"Publish evidence bundle {bundle_id} for {req.control_id}.",
            requested_by_run_id=run_id,
        )
    )

    # A real system would insert this row into the journal table. Here we return
    # it so readers can see what should be recorded.
    journal_row = {
        "run_id": run_id,
        "tool_name": "publish_evidence_bundle",
        "idempotency_key": publish_key,
        "status": "approved" if approval["decision"] == "approved" else "blocked",
        "approval_reference": approval["request_id"],
        "requested_at": datetime.now(timezone.utc).isoformat(),
    }

    return {
        "request": req.model_dump(),
        "bundle_status": draft_status.model_dump(),
        "approval": approval,
        "journal_row": journal_row,
    }

request = EvidenceRequest(
    control_id="CTRL-77",
    period="2026-Q2",
    jurisdiction="uk",
    requested_by_user_id="owner-14",
)

await assemble_evidence_bundle(request)


## 9. Notebook self-check

Run this after executing the notebook from the top. It checks the important learning examples: tool selection, preview without side effects, idempotent replay, approval token, MCP registration, and the audit-prep workflow output.


In [ ]:
# The self-check is here for readers. If this cell passes, the core examples are
# wired correctly and the notebook is in a good state.
checks = {
    "tool_selection_prefers_profile": ranked_tools[0][0] == "get_user_profile",
    "po_preview_has_no_side_effect": preview["side_effect"] is False,
    "refund_retry_replays_same_refund": (
        first_attempt["replayed"] is False
        and retry_attempt["replayed"] is True
        and first_attempt["refund_id"] == retry_attempt["refund_id"]
    ),
    "approval_returns_token": bool(approval.get("approval_token")),
    "mcp_tool_registered": "get_trial_balance_snapshot" in app.tools,
    "audit_agent_has_three_servers": len(agent.mcp_servers) == 3,
}

if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise AssertionError(f"Notebook self-check failed: {failed}")

checks


## 10. Optional live API smoke test

Run this cell only when you want to confirm that your OpenAI API key and the selected model work from the notebook environment. It reads `OPENAI_API_KEY` from your environment and uses `gpt-5-mini` by default. It never stores or prints the key.


In [ ]:
import os

# Keep the model name in one place so readers can change it deliberately.
# The requested Chapter 3 live check uses GPT-5 mini.
LIVE_MODEL = os.getenv("OPENAI_LIVE_MODEL", "gpt-5-mini")

if not os.getenv("OPENAI_API_KEY"):
    live_smoke_result = {
        "skipped": True,
        "reason": "Set OPENAI_API_KEY to run the live API smoke test.",
        "model": LIVE_MODEL,
    }
else:
    try:
        from openai import OpenAI
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "Install the Chapter 3 requirements before running the live API smoke test: "
            "python -m pip install -r requirements.txt"
        ) from exc

    client = OpenAI()
    response = client.responses.create(
        model=LIVE_MODEL,
        input="Reply exactly with: chapter-3-live-smoke-ok",
        reasoning={"effort": "minimal"},
        max_output_tokens=256,
    )

    live_output = response.output_text.strip()
    live_smoke_result = {
        "skipped": False,
        "model": LIVE_MODEL,
        "output": live_output,
    }

    if "chapter-3-live-smoke-ok" not in live_output.lower():
        raise AssertionError(f"Unexpected live smoke-test output: {live_output!r}")

live_smoke_result


## 11. Mapping notebook sections back to the chapter

- `Understanding what a tool actually is`: function plus description plus schema.
- `How the agent picks the right tool`: decision reliability and tool descriptions.
- `Designing tools the model can actually use`: intent-shaped tools, tight schemas, and useful errors.
- `Engineering the side-effect boundary`: idempotency, preview/execute, approval gates, and compensation.
- `Plugging in with the Model Context Protocol`: MCP topology, server primitives, and transport choice.
- `Building a financial services audit-prep agent`: tool surface, journal, approval gate, and end-to-end workflow.

The practical takeaway: Chapter 3 is where the Tool Registry and Constraint Boundary become concrete engineering surfaces. The agent gets hands, but every hand has a contract, an audit trail, and a safety boundary.
